In [ ]:
# @title Data Acquisition Module
import os
import re
import string
import nltk
from nltk.corpus import stopwords
import kagglehub
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

def process_dataset():
  print("Downloading nltk and English stopwords...")
  nltk.download('stopwords')
  english_stopwords = set(stopwords.words('english'))

  print("Downloading fake reviews dataset...")
  dataset_path = kagglehub.dataset_download("muqaddasejaz/fake-reviews-dataset")
  csv_path = os.path.join(dataset_path, "fake reviews dataset.csv")
  print(f"Dataset downloaded.")

  dataset_frame = pd.read_csv(csv_path)

  print(f"Original dataset shape: {dataset_frame.shape}")
  print(dataset_frame[['label', 'text_']].head())

  print("Pre-processing the downloaded dataset...")
  dataset_frame['clean_text'] = dataset_frame['text_'].apply(lambda x: clean_dataset(x, english_stopwords))
  dataset_frame['binary_label'] = dataset_frame['label'].map({'CG': 1, 'OR': 0})
  dataset_frame = dataset_frame[dataset_frame['clean_text'].str.strip() != ""]
  print("Dataset pre-processing completed.")

  print(f"Pre-processed dataset shape: {dataset_frame.shape}")
  print(dataset_frame[['label', 'binary_label', 'text_', 'clean_text']].head())

  print("Exporting cleaned dataset...")
  export_filename = 'datasets/cleaned_fake_reviews.csv'
  dataset_frame.to_csv(export_filename, index=False)
  print(f".csv file exported as '{export_filename}'.")

  return dataset_frame

def clean_dataset(text, english_stopwords):
  if not isinstance(text, str):
    return ""

  print("Performing lowercase conversion...")
  text = text.lower()
  print("Performing punctuation and numbers removal...")
  text = text.translate(str.maketrans('', '', string.punctuation))
  text = re.sub(r'\d+', '', text)
  print("Performing stopwords removal...")
  words = text.split()
  words = [w for w in words if w not in english_stopwords]
  print("Cleaning process finished.")

  return ' '.join(words)

def train_test_split_dataset(dataset_frame):
  X = dataset_frame['clean_text'].values
  Y = dataset_frame['binary_label'].values

  X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=59)
  print(f"Training set: {len(X_train)} samples")
  print(f"Test set: {len(X_test)} samples")

  return X_train, X_test, Y_train, Y_test

def vectorize_dataset(X_train, X_test):
  tfidf_vectorizer = TfidfVectorizer(max_features=10000)
  X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
  X_test_tfidf = tfidf_vectorizer.transform(X_test)

  print(f"Shape of training set: {X_train_tfidf.shape}")
  print(f"Shape of test set: {X_test_tfidf.shape}")

  with open('models/tfidf_vectorizer.pkl', 'wb') as bfw:
    pickle.dump(tfidf_vectorizer, bfw)

  return X_train_tfidf, X_test_tfidf

def tokenize_dataset(X_train, X_test):
  max_words = 10000
  lstm_tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
  lstm_tokenizer.fit_on_texts(X_train)

  X_train_sequences = lstm_tokenizer.texts_to_sequences(X_train)
  X_test_sequences = lstm_tokenizer.texts_to_sequences(X_test)

  max_sequence_length = 200
  X_train_padded = pad_sequences(X_train_sequences, maxlen=max_sequence_length, padding='post', truncating='post')
  X_test_padded = pad_sequences(X_test_sequences, maxlen=max_sequence_length, padding='post', truncating='post')

  print(f"Shape of training set: {X_train_padded.shape}")
  print(f"Shape of test set: {X_test_padded.shape}")

  with open('models/lstm_tokenizer.pkl', 'wb') as bfw:
    pickle.dump(lstm_tokenizer, bfw)

  return X_train_padded, X_test_padded

print("Downloading and processing dataset...")
dataset_frame = process_dataset()
print("Dataset is fully processed.")

print("Splitting dataset into training and test sets...")
dataset_X_train, dataset_X_test, dataset_Y_train, dataset_Y_test = train_test_split_dataset(dataset_frame)
print("Dataset split into training and test sets.")

print("Vectorizing training/test sets and exporting...")
X_train_tfidf, X_test_tfidf = vectorize_dataset(dataset_X_train, dataset_X_test)

print("Tokenizing training/test sets and exporting...")
X_train_padded, X_test_padded = tokenize_dataset(dataset_X_train, dataset_X_test)
print("Exports saved and relevant files are available for download.")

In [ ]:
# @title Naive-Bayes Training and Testing
import time
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split, KFold
from sklearn.naive_bayes import MultinomialNB

def load_dataset_nb():
  print("Loading cleaned fake reviews dataset...")
  dataset_frame = pd.read_csv('datasets/cleaned_fake_reviews.csv')
  dataset_x_train, dataset_x_test, dataset_y_train, dataset_y_test = re_split_train_test_dataset_nb(dataset_frame)
  return dataset_x_train, dataset_x_test, dataset_y_train, dataset_y_test

def re_split_train_test_dataset_nb(dataset_frame):
  X = dataset_frame['clean_text'].fillna("").values
  Y = dataset_frame['binary_label'].values

  print("Splitting the dataset to training/test sets...")
  X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=59)
  print(f"Training set: {len(X_train)} samples")
  print(f"Test set: {len(X_test)} samples")

  return X_train, X_test, Y_train, Y_test

def vectorize_dataset_nb(X_train, X_test):
  with open('models/tfidf_vectorizer.pkl', 'rb') as bfr:
    tfidf_vectorizer = pickle.load(bfr)

  print("Transforming the text in the training/test sets...")
  X_train_tfidf = tfidf_vectorizer.transform(X_train)
  X_test_tfidf = tfidf_vectorizer.transform(X_test)
  print("Training/test set texts are transformed and matrices are formed.")

  return X_train_tfidf, X_test_tfidf

def train_test_naive_bayes_model(X_train_tfidf, X_test_tfidf, dataset_y_train, dataset_y_test):
  naive_bayes_classifier = MultinomialNB(alpha=1.0)

  print("Initializing the training of the Naive-Bayes based model...")
  start_train_time = time.time()
  naive_bayes_classifier.fit(X_train_tfidf, dataset_y_train)
  train_time = time.time() - start_train_time
  print(f"Training completed in {train_time:.4f} seconds.")

  print("Initializing the testing of the Naive-Bayes based model...")
  start_infer_time = time.time()
  y_prediction_naive_bayes = naive_bayes_classifier.predict(X_test_tfidf)
  infer_time = time.time() - start_infer_time
  print(f"Inference completed in {infer_time:.4f} seconds.")

  nb_pred_accuracy = accuracy_score(dataset_y_test, y_prediction_naive_bayes)
  nb_pred_precision = precision_score(dataset_y_test, y_prediction_naive_bayes)
  nb_pred_recall = recall_score(dataset_y_test, y_prediction_naive_bayes)
  nb_pred_f1 = f1_score(dataset_y_test, y_prediction_naive_bayes)

  print("--- Naive-Bayes Performance Metrics ---")
  print(f"Accuracy:  {nb_pred_accuracy:.4f}")
  print(f"Precision: {nb_pred_precision:.4f}")
  print(f"Recall:    {nb_pred_recall:.4f}")
  print(f"F1-Score:  {nb_pred_f1:.4f}")

  with open('models/naive_bayes_model.pkl', 'wb') as bfr:
    pickle.dump(naive_bayes_classifier, bfr)
    print("Model saved as 'naive_bayes_model.pkl'.")

  return y_prediction_naive_bayes

def k_fold_cross_validation_nb(X_train_tfidf, dataset_y_train):
  print("Performing 5-Fold Cross-Validation...")
  kfold = KFold(n_splits=5, shuffle=True, random_state=59)
  cross_val_scores_nb = []

  fold = 1
  for train_index, val_index in kfold.split(X_train_tfidf):
    X_fold_train, X_fold_val = X_train_tfidf[train_index], X_train_tfidf[val_index]
    y_fold_train, y_fold_val = dataset_y_train[train_index], dataset_y_train[val_index]

    fold_model = MultinomialNB(alpha=1.0)
    fold_model.fit(X_fold_train, y_fold_train)

    y_prediction_fold = fold_model.predict(X_fold_val)
    accuracy_fold = accuracy_score(y_fold_val, y_prediction_fold)
    cross_val_scores_nb.append(accuracy_fold)
    print(f"Fold {fold} Validation Accuracy: {accuracy_fold:.4f}")
    fold += 1

  print(f"Cross-Validation Accuracy Scores: {cross_val_scores_nb}")
  print(f"Mean CV Accuracy: {np.mean(cross_val_scores_nb):.4f} (+/- {np.std(cross_val_scores_nb) * 2:.4f})")

  return cross_val_scores_nb

def generate_confusion_matrix_nb(dataset_y_true, y_prediction):
  conf_matrix = confusion_matrix(dataset_y_true, y_prediction)
  plt.figure(figsize=(6, 4))
  sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Original (0)', 'Computer Generated (1)'], yticklabels=['Original (0)', 'Computer Generated (1)'])
  plt.title('Naive-Bayes Confusion Matrix')
  plt.xlabel('Predicted Label')
  plt.ylabel('True Label')
  plt.tight_layout()
  plt.savefig('naive_bayes_confusion_matrix.png')
  print("Confusion matrix saved as 'naive_bayes_confusion_matrix.png'.")
  plt.show()

dataset_x_train, dataset_x_test, dataset_y_train, dataset_y_test = load_dataset_nb()
X_train_tfidf, X_test_tfidf = vectorize_dataset_nb(dataset_x_train, dataset_x_test)
cross_val_scores_nb = k_fold_cross_validation_nb(X_train_tfidf, dataset_y_train)
y_prediction_nb = train_test_naive_bayes_model(X_train_tfidf, X_test_tfidf, dataset_y_train, dataset_y_test)
generate_confusion_matrix_nb(dataset_y_test, y_prediction_nb)

In [ ]:
# @title k-NN Training and Testing
import pandas as pd
import pickle
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.decomposition import TruncatedSVD

def load_dataset_knn():
  print("Loading cleaned fake reviews dataset...")
  dataset_frame = pd.read_csv('datasets/cleaned_fake_reviews.csv')
  dataset_x_train, dataset_x_test, dataset_y_train, dataset_y_test = re_split_train_test_dataset_knn(dataset_frame)
  return dataset_x_train, dataset_x_test, dataset_y_train, dataset_y_test

def re_split_train_test_dataset_knn(dataset_frame):
  X = dataset_frame['clean_text'].fillna("").values
  Y = dataset_frame['binary_label'].values

  print("Splitting the dataset to training/test sets...")
  X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=59)
  print(f"Training set: {len(X_train)} samples")
  print(f"Test set: {len(X_test)} samples")

  return X_train, X_test, Y_train, Y_test

def vectorize_dataset_knn(X_train, X_test):
  with open('models/tfidf_vectorizer.pkl', 'rb') as bfr:
    tfidf_vectorizer = pickle.load(bfr)

  print("Transforming the text in the training/test sets...")
  X_train_tfidf = tfidf_vectorizer.transform(X_train)
  X_test_tfidf = tfidf_vectorizer.transform(X_test)
  print("Training/test set texts are transformed and matrices are formed.")

  ### TruncatedSVD application - comment to get pure k-NN results.
  dimension_count = 10000
  """
  print("Applying TruncatedSVD to reduce dimensions...")
  truncated_svd = TruncatedSVD(n_components=dimension_count, random_state=59)
  X_train_tfidf = truncated_svd.fit_transform(X_train_tfidf)
  X_test_tfidf = truncated_svd.transform(X_test_tfidf)
  print("Dimensions reduced.")

  with open('models/knn_svd_transformer.pkl', 'wb') as bfr:
      pickle.dump(truncated_svd, bfr)
      print("SVD transformer saved as 'knn_svd_transformer.pkl'.")
  """
  ### TruncatedSVD ends.

  return X_train_tfidf, X_test_tfidf, dimension_count

def train_test_knn_model(X_train_tfidf, X_test_tfidf, dataset_y_train, dataset_y_test, set_dimensions):
  # Optimal Value (100 Dimensions): 9
  # Optimal Value (10000 Dimensions): 2
  neighbor_count = 2
  knn_classifier = KNeighborsClassifier(n_neighbors=neighbor_count)

  print("Initializing the training of the k-NN based model...")
  start_train_time = time.time()
  knn_classifier.fit(X_train_tfidf, dataset_y_train)
  train_time = time.time() - start_train_time
  print(f"Training completed in {train_time:.4f} seconds.")

  print("Initializing the testing of the k-NN based model...")
  start_infer_time = time.time()
  y_prediction_knn = knn_classifier.predict(X_test_tfidf)
  infer_time = time.time() - start_infer_time
  print(f"Inference completed in {infer_time:.4f} seconds.")

  knn_pred_accuracy = accuracy_score(dataset_y_test, y_prediction_knn)
  knn_pred_precision = precision_score(dataset_y_test, y_prediction_knn)
  knn_pred_recall = recall_score(dataset_y_test, y_prediction_knn)
  knn_pred_f1 = f1_score(dataset_y_test, y_prediction_knn)

  print("--- k-NN Performance Metrics ---")
  print(f"Accuracy:  {knn_pred_accuracy:.4f}")
  print(f"Precision: {knn_pred_precision:.4f}")
  print(f"Recall:    {knn_pred_recall:.4f}")
  print(f"F1-Score:  {knn_pred_f1:.4f}")

  with open(f'models/knn_model_{set_dimensions}_features.pkl', 'wb') as bfr:
      pickle.dump(knn_classifier, bfr)
      print(f"Model saved as 'knn_model_{set_dimensions}_features.pkl', and is available for download.")

  return y_prediction_knn, neighbor_count

def k_fold_cross_validation_knn(X_train_tfidf, dataset_y_train):
  print("Performing 5-Fold Cross-Validation...")
  kfold = KFold(n_splits=5, shuffle=True, random_state=59)
  cross_val_scores_knn = []

  fold = 1
  for train_index, val_index in kfold.split(X_train_tfidf):
    X_fold_train, X_fold_val = X_train_tfidf[train_index], X_train_tfidf[val_index]
    y_fold_train, y_fold_val = dataset_y_train[train_index], dataset_y_train[val_index]

    fold_model = KNeighborsClassifier(n_neighbors=9) # Make it 9 for 100 dimensional training
    fold_model.fit(X_fold_train, y_fold_train)

    y_prediction_fold = fold_model.predict(X_fold_val)
    accuracy_fold = accuracy_score(y_fold_val, y_prediction_fold)
    cross_val_scores_knn.append(accuracy_fold)
    print(f"Fold {fold} Validation Accuracy: {accuracy_fold:.4f}")
    fold += 1

  print(f"Cross-Validation Accuracy Scores: {cross_val_scores_knn}")
  print(f"Mean CV Accuracy: {np.mean(cross_val_scores_knn):.4f} (+/- {np.std(cross_val_scores_knn) * 2:.4f})")

  return cross_val_scores_knn

def generate_confusion_matrix_knn(dataset_y_true, y_prediction, neighbors, dimensions):
  conf_matrix = confusion_matrix(dataset_y_true, y_prediction)
  plt.figure(figsize=(6, 4))
  sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Original (0)', 'Computer Generated (1)'], yticklabels=['Original (0)', 'Computer Generated (1)'])
  plt.title(f'k-NN Confusion Matrix ({neighbors} Neighbors, {dimensions} Dimensions)')
  plt.xlabel('Predicted Label')
  plt.ylabel('True Label')
  plt.tight_layout()
  plt.savefig(f'knn_confusion_matrix_{neighbors}_neighbors_{dimensions}_dimensions.png')
  print(f"Confusion matrix is saved as 'knn_confusion_matrix_{neighbors}_neighbors_{dimensions}_dimensions.png'.")
  plt.show()

dataset_x_train, dataset_x_test, dataset_y_train, dataset_y_test = load_dataset_knn()
X_train_tfidf, X_test_tfidf, set_dimensions = vectorize_dataset_knn(dataset_x_train, dataset_x_test)
cross_val_scores_knn = k_fold_cross_validation_knn(X_train_tfidf, dataset_y_train)
y_prediction_knn, knn_neighbor_count = train_test_knn_model(X_train_tfidf, X_test_tfidf, dataset_y_train, dataset_y_test, set_dimensions)
generate_confusion_matrix_knn(dataset_y_test, y_prediction_knn, knn_neighbor_count, set_dimensions)

In [ ]:
# @title LSTM Training and Testing
import os
import pandas as pd
import pickle
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def load_dataset_lstm():
  print("Loading cleaned fake reviews dataset...")
  dataset_frame = pd.read_csv('datasets/cleaned_fake_reviews.csv')
  dataset_x_train, dataset_x_test, dataset_y_train, dataset_y_test = re_split_train_test_dataset_lstm(dataset_frame)
  return dataset_x_train, dataset_x_test, dataset_y_train, dataset_y_test

def re_split_train_test_dataset_lstm(dataset_frame):
  X = dataset_frame['clean_text'].fillna("").values
  Y = dataset_frame['binary_label'].values

  print("Splitting the dataset to training/test sets...")
  X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=59)
  print(f"Training set: {len(X_train)} samples")
  print(f"Test set: {len(X_test)} samples")

  return X_train, X_test, Y_train, Y_test

def tokenize_and_pad_lstm(X_train, X_test):
  with open('models/lstm_tokenizer.pkl', 'rb') as bfr:
    lstm_tokenizer = pickle.load(bfr)

  print("Converting text to integer sequences...")
  X_train_seq = lstm_tokenizer.texts_to_sequences(X_train)
  X_test_seq = lstm_tokenizer.texts_to_sequences(X_test)

  print("Padding sequences to exactly 200 words...")
  X_train_pad = pad_sequences(X_train_seq, maxlen=200, padding='pre', truncating='post')
  X_test_pad = pad_sequences(X_test_seq, maxlen=200, padding='pre', truncating='post')

  print(f"LSTM Training matrix shape: {X_train_pad.shape}\n")
  return X_train_pad, X_test_pad

def train_test_lstm_model(X_train_pad, X_test_pad, dataset_y_train, dataset_y_test):
  print("Building the LSTM based model...")
  lstm_model = Sequential()
  lstm_model.add(Embedding(input_dim=10000, output_dim=64))
  lstm_model.add(Bidirectional(LSTM(64, dropout=0.2)))
  lstm_model.add(Dense(1, activation='sigmoid'))
  custom_adam = Adam(learning_rate=0.001)
  lstm_model.compile(optimizer=custom_adam, loss='binary_crossentropy', metrics=['accuracy'])
  print(lstm_model.summary())

  early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

  print("Initializing the training of the LSTM based model...")
  start_train_time = time.time()
  history = lstm_model.fit(X_train_pad, dataset_y_train, epochs=3, batch_size=32, validation_split=0.1, callbacks=[early_stopping])
  train_time = time.time() - start_train_time
  print(f"Training completed in {train_time:.4f} seconds.")

  print("Initializing the testing of the LSTM based model...")
  start_infer_time = time.time()
  y_pred_probabilities = lstm_model.predict(X_test_pad)
  y_prediction_lstm = (y_pred_probabilities > 0.5).astype(int).flatten()
  infer_time = time.time() - start_infer_time
  print(f"Inference completed in {infer_time:.4f} seconds.")

  lstm_pred_accuracy = accuracy_score(dataset_y_test, y_prediction_lstm)
  lstm_pred_precision = precision_score(dataset_y_test, y_prediction_lstm)
  lstm_pred_recall = recall_score(dataset_y_test, y_prediction_lstm)
  lstm_pred_f1 = f1_score(dataset_y_test, y_prediction_lstm)

  print("--- LSTM Performance Metrics ---")
  print(f"Accuracy:  {lstm_pred_accuracy:.4f}")
  print(f"Precision: {lstm_pred_precision:.4f}")
  print(f"Recall:    {lstm_pred_recall:.4f}")
  print(f"F1-Score:  {lstm_pred_f1:.4f}")

  lstm_model.save('models/lstm_model.keras')
  print("Model saved as 'lstm_model.keras'.")

  return y_prediction_lstm

def k_fold_cross_validation_lstm(X_train_pad, dataset_y_train):
  print("Performing 5-Fold Cross-Validation...")
  kfold = KFold(n_splits=5, shuffle=True, random_state=59)
  cross_val_scores = []

  fold = 1
  for train_index, valid_index in kfold.split(X_train_pad):
    print(f"\n--- Training Fold {fold} ---")
    X_fold_train, X_fold_val = X_train_pad[train_index], X_train_pad[valid_index]
    y_fold_train, y_fold_val = dataset_y_train[train_index], dataset_y_train[valid_index]

    fold_model = Sequential()
    fold_model.add(Embedding(input_dim=10000, output_dim=64, input_length=200))
    fold_model.add(Bidirectional(LSTM(64, dropout=0.2)))
    fold_model.add(Dense(1, activation='sigmoid'))

    fold_adam = Adam(learning_rate=0.001)
    fold_model.compile(optimizer=fold_adam, loss='binary_crossentropy', metrics=['accuracy'])

    fold_early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    fold_model.fit(X_fold_train, y_fold_train, epochs=3, batch_size=32, validation_data=(X_fold_val, y_fold_val), callbacks=[fold_early_stopping])
    fold_val_loss, fold_val_accuracy = fold_model.evaluate(X_fold_val, y_fold_val)
    cross_val_scores.append(fold_val_accuracy)
    print(f"Fold {fold} Validation Accuracy: {fold_val_accuracy:.4f}")
    fold += 1

  print(f"Cross-Validation Accuracy Scores: {cross_val_scores}")
  print(f"Mean CV Accuracy: {np.mean(cross_val_scores):.4f} (+/- {np.std(cross_val_scores) * 2:.4f})")

  return cross_val_scores

def generate_confusion_matrix_lstm(dataset_y_true, y_prediction):
  conf_matrix = confusion_matrix(dataset_y_true, y_prediction)
  plt.figure(figsize=(6, 4))
  sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Original (0)', 'Computer Generated (1)'], yticklabels=['Original (0)', 'Computer Generated (1)'])
  plt.title('LSTM Confusion Matrix')
  plt.xlabel('Predicted Label')
  plt.ylabel('True Label')
  plt.tight_layout()
  plt.savefig('lstm_confusion_matrix.png')
  print("Confusion matrix saved as 'lstm_confusion_matrix.png'.")
  plt.show()

dataset_x_train, dataset_x_test, dataset_y_train, dataset_y_test = load_dataset_lstm()
X_train_padded, X_test_padded = tokenize_and_pad_lstm(dataset_x_train, dataset_x_test)
cross_val_scores_lstm = k_fold_cross_validation_lstm(X_train_padded, dataset_y_train)
y_prediction_lstm = train_test_lstm_model(X_train_padded, X_test_padded, dataset_y_train, dataset_y_test)
generate_confusion_matrix_lstm(dataset_y_test, y_prediction_lstm)

In [ ]:
#@title Plot Generator for the Project Report
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def generate_comparison_bar_chart():
  model_metrics_data = {
    'Base Algorithm': ['Naive-Bayes', 'Naive-Bayes', 'Naive-Bayes', 'Naive-Bayes', 'Naive-Bayes',
      'k-NN (10k Features)', 'k-NN (10k Features)', 'k-NN (10k Features)', 'k-NN (10k Features)', 'k-NN (10k Features)',
      'k-NN (100 Features)', 'k-NN (100 Features)', 'k-NN (100 Features)', 'k-NN (100 Features)', 'k-NN (100 Features)',
      'Bidirectional-LSTM', 'Bidirectional-LSTM', 'Bidirectional-LSTM', 'Bidirectional-LSTM', 'Bidirectional-LSTM'],
    'Metric': ['Mean Cross-Val Acc.', 'Accuracy', 'Precision', 'Recall', 'F1-Score'] * 4,
    'Score': [
      0.8578, 0.8610, 0.8403, 0.8921, 0.8654,  # Naive-Bayes
      0.6361, 0.7168, 0.6740, 0.8418, 0.7486,  # k-NN (10k Features)
      0.7546, 0.7622, 0.8697, 0.6179, 0.7225,  # k-NN (100 Features)
      0.9121, 0.9172, 0.9126, 0.9230, 0.9178   # LSTM
      ]
    }

  metrics_dataframe = pd.DataFrame(model_metrics_data)
  plt.figure(figsize=(12, 6))
  sns.barplot(x='Metric', y='Score', hue='Base Algorithm', palette='crest', data=metrics_dataframe)
  plt.title('Performance Comparison Across All Models', fontsize=16)
  plt.ylim(0, 1.1)
  plt.ylabel('Score')
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

  plt.tight_layout()
  plt.savefig('comparison_bar_chart.png', dpi=300)
  print("Comparison bar chart is saved as 'comparison_bar_chart.png'.")
  plt.show()


def generate_confusion_matrix_combined():
  figure, axes = plt.subplots(2, 2, figsize=(12, 10))

  try:
    img_nb = mpimg.imread('naive_bayes_confusion_matrix.png')
    img_knn_stock = mpimg.imread('knn_confusion_matrix_2_neighbors_10000_dimensions.png')
    img_knn_svd = mpimg.imread('knn_confusion_matrix_9_neighbors_100_dimensions.png')
    img_lstm = mpimg.imread('lstm_confusion_matrix.png')

    axes[0, 0].imshow(img_nb)
    axes[0, 0].axis('off')

    axes[0, 1].imshow(img_knn_stock)
    axes[0, 1].axis('off')

    axes[1, 0].imshow(img_knn_svd)
    axes[1, 0].axis('off')

    axes[1, 1].imshow(img_lstm)
    axes[1, 1].axis('off')

    plt.tight_layout()
    plt.savefig('confusion_matrix_combined.png', dpi=300)
    print("All confusion matrixes are combined into the 'confusion_matrix_combined.png' file.")
    plt.show()

  except Exception as ex:
    print(f"Error loading images: {ex}")

generate_comparison_bar_chart()
generate_confusion_matrix_combined()